In [ ]:
import datetime

DATE = datetime.datetime.now()

# comment out line above and use line below if you want to fetch the data from a specific date
# Replace YYYY-MM-DD with your date in the same format
# DATE = datetime.datetime.fromisoformat("YYYY-MM-DD")

### Load api key from .env file into os.environ, then read env variable from there

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads variables from .env into os.environ

API_KEY = os.getenv("FIRM_API_KEY")
if not API_KEY:
    raise Exception("Missing FIRM_API_KEY. Set it in your environment or .env file.")

### Test api key

In [ ]:
import requests

url = "https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY="
parameters = {"MAP_KEY": API_KEY}
response = requests.get(url, params=parameters)
if response.status_code == 200:
    print(f"Api request successful\n{'-' * 22}")

    # Convert the JSON response into a Python dictionary
    data = response.json()
    print(f"Current transactions: {data['current_transactions']}/5000")
else:
    raise Exception(
        f"An error occured with status code {response.status_code}: {response.reason}"
    )

### Fetch data from the last 14 days

In [ ]:
import time
import requests
from pathlib import Path

date = DATE + datetime.timedelta(days=-1)
date_str = date.strftime("%Y-%m-%d")
data = []

for _i in range(0, 7):
    url = (
        "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        + API_KEY
        + "/VIIRS_SNPP_NRT/world/2/"
        + date_str
    )
    date = date + datetime.timedelta(days=-2)
    date_str = date.strftime("%Y-%m-%d")
    response = requests.get(url)
    if response.status_code == 200:
        parsed_response = response.content.decode("utf-8")
        # if data is still empty, also append the column
        if not data:
            data.extend(parsed_response.split("\n"))
        else:
            data.extend(parsed_response.split("\n")[1:])
    else:
        raise Exception(
            f"An error occured with status code {response.status_code}: {response.reason}"
        )
    # api seems to behave well, but we sleep half second nonetheless
    time.sleep(0.5)
# use Path object so we don't have to worry about path separator differences
file_path = Path.cwd().parent.joinpath(
    "data", f"VIIRS_SNNP_NRT_world_14days_{DATE.strftime('%Y-%m-%d')}.csv"
)

with open(file_path, "w") as file:
    file.write("\n".join(data))
